# KBO 가을야구 예측 전처리 실험 - dy

이 노트북은 `data/raw`와 `data/processed`에 있는 2022~2026 KBO 데이터를 기준으로 전처리, 예측, 시각화 산출물을 다시 생성하는 실행용 파일입니다.

산출물은 모두 `notebooks/experiments/dy` 폴더 안에 저장됩니다.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

EXPERIMENT_DIR = Path(r"C:\\Users\\Admin\\Documents\\GitHub\\Ml_Baseball\\notebooks\\experiments\\dy")
REPO_DIR = EXPERIMENT_DIR.parents[2]
RAW_DIR = REPO_DIR / "data" / "raw"
PROCESSED_DIR = REPO_DIR / "data" / "processed"

sys.path.insert(0, str(EXPERIMENT_DIR))

print("실험 폴더:", EXPERIMENT_DIR)
print("원본 데이터:", RAW_DIR)
print("과정 데이터:", PROCESSED_DIR)


실험 폴더: C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy
원본 데이터: C:\Users\Admin\Documents\GitHub\Ml_Baseball\data\raw
과정 데이터: C:\Users\Admin\Documents\GitHub\Ml_Baseball\data\processed


## 1. 입력 데이터 확인

사용 데이터 범위는 2022~2026입니다. 2022~2025는 완료 시즌, 2026은 진행 중 시즌 데이터로 다룹니다.


In [2]:
for base in [RAW_DIR, PROCESSED_DIR]:
    print(f"\n== {base} ==")
    for year in range(2022, 2027):
        files = sorted((base / str(year)).glob("*.csv"))
        print(year, len(files), "files")



== C:\Users\Admin\Documents\GitHub\Ml_Baseball\data\raw ==
2022 12 files
2023 12 files
2024 12 files
2025 12 files
2026 13 files

== C:\Users\Admin\Documents\GitHub\Ml_Baseball\data\processed ==
2022 9 files
2023 9 files
2024 9 files
2025 9 files
2026 7 files


## 2. 전체 파이프라인 실행

아래 셀은 raw/processed 데이터를 읽어서 모델 데이터셋, 2026 예측 결과, 검증 결과, 발표용 PNG, HTML 대시보드를 한 번에 생성합니다.


In [3]:
import dy_experiment_pipeline as pipe

summary = pipe.run_all()
summary


[saved] C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy\01_league_trend.png
[saved] C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy\02_rank_heatmap.png
[saved] C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy\03_era_winrate.png
[saved] C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy\04_playoff_boxplot.png
[saved] C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy\05_transfer.png
[saved] C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy\06_feature_importance.png
[saved] C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy\07_playoff_prob.png
[saved] C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy\08_apr_vs_final.png
[saved] C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy\09_cluster_pca.png
Generated presentation PNG files in C:\Users\Admin\Documents\GitHub\Ml_Baseball\notebooks\experiments\dy
C:\Users\Admin\Docu

{'input_raw_dir': 'C:\\Users\\Admin\\Documents\\GitHub\\Ml_Baseball\\data\\raw',
 'input_processed_dir': 'C:\\Users\\Admin\\Documents\\GitHub\\Ml_Baseball\\data\\processed',
 'output_dir': 'C:\\Users\\Admin\\Documents\\GitHub\\Ml_Baseball\\notebooks\\experiments\\dy',
 'years': [2022, 2023, 2024, 2025, 2026],
 'training_rows': 6125,
 'latest_2026_date': '2026-04-26',
 'predicted_top5': ['LG', 'KT', 'SSG', '삼성', 'KIA'],
 'generated_pngs': ['01_league_trend.png',
  '02_rank_heatmap.png',
  '03_era_winrate.png',
  '04_playoff_boxplot.png',
  '05_transfer.png',
  '06_feature_importance.png',
  '07_playoff_prob.png',
  '08_apr_vs_final.png',
  '09_cluster_pca.png']}

## 3. 모델 학습용 데이터 확인

`kbo_outputs/model_dataset_2022_2026.csv`는 예측에 사용되는 최종 학습/예측용 스냅샷 데이터입니다.


In [4]:
model_df = pd.read_csv(EXPERIMENT_DIR / "kbo_outputs" / "model_dataset_2022_2026.csv", encoding="utf-8-sig")
print(model_df.shape)
model_df[["season", "date", "team", "rank", "win_rate", "postseason", "source"]].head()


(6770, 123)


,season,date,team,rank,win_rate,postseason,source
0,2022,2022-04-02,KIA,6,0.000,1.0,processed/train_dataset_2022
1,2022,2022-04-03,KIA,8,0.000,1.0,processed/train_dataset_2022
2,2022,2022-04-05,KIA,6,0.333,1.0,processed/train_dataset_2022
3,2022,2022-04-06,KIA,4,0.500,1.0,processed/train_dataset_2022
4,2022,2022-04-07,KIA,3,0.600,1.0,processed/train_dataset_2022


## 4. 2026 가을야구 예측 결과

현재 진행 중인 2026 시즌 기준 팀별 가을야구 진출 확률입니다.


In [5]:
pred = pd.read_csv(EXPERIMENT_DIR / "kbo_outputs" / "2026_postseason_predictions.csv", encoding="utf-8-sig")
pred[["team", "rank", "wins", "losses", "win_rate", "postseason_probability_pct", "prediction_label"]]


,team,rank,wins,losses,win_rate,postseason_probability_pct,prediction_label
0,LG,2,16,8,0.667,96.1,진출
1,KT,1,17,8,0.680,93.3,진출
2,SSG,3,15,9,0.625,90.3,진출
3,삼성,4,12,11,0.522,53.9,진출
4,KIA,5,12,12,0.500,44.3,진출
5,NC,6,11,13,0.458,30.5,미진출
6,한화,7,10,14,0.417,30.1,미진출
7,두산,7,10,14,0.417,26.2,미진출
8,롯데,10,7,16,0.304,5.8,미진출
9,키움,9,10,15,0.400,4.4,미진출


## 5. 검증 및 4월 인사이트

완료 시즌 기준 검증 결과와 4월 순위가 최종 순위와 얼마나 맞았는지 확인합니다.


In [6]:
validation = pd.read_csv(EXPERIMENT_DIR / "kbo_outputs" / "validation_leave_one_season.csv", encoding="utf-8-sig")
april = pd.read_csv(EXPERIMENT_DIR / "kbo_outputs" / "april_rank_insight.csv", encoding="utf-8-sig")
display(validation)
display(april)


,season,rows,auc,latest_top5_overlap,latest_top5_overlap_rate
0,2022,1496,0.881,4,0.8
1,2023,1566,0.938,5,1.0
2,2024,1530,0.906,5,1.0
3,2025,1533,0.732,4,0.8


,season,april_latest_date,current_top5_overlap,current_top5_overlap_rate,rank_final_spearman
0,2023,2023-04-30,4,0.8,0.413
1,2024,2024-04-30,3,0.6,0.418
2,2025,2025-04-30,3,0.6,0.685


## 6. 최종 산출물 위치

아래 파일들이 발표/제출용 결과입니다.

- `2026_가을야구_예측결과.csv`
- `team_master_2022_2026.csv`
- `리그_타격환경.csv`, `리그_투구환경.csv`
- `01_*.png` ~ `09_*.png`
- `kbo_2022_2026_master_dashboard.html`


In [7]:
for p in sorted(EXPERIMENT_DIR.glob("*.csv")):
    print(p.name, p.stat().st_size)

print("\nPNG files")
for p in sorted(EXPERIMENT_DIR.glob("0*.png")):
    print(p.name, p.stat().st_size)

html_path = EXPERIMENT_DIR / "kbo_2022_2026_master_dashboard.html"
print("\nHTML:", "file:///" + str(html_path).replace("\\", "/"))


2026_가을야구_예측결과.csv 556
team_master_2022_2026.csv 14193
리그_타격환경.csv 201
리그_투구환경.csv 323

PNG files
01_league_trend.png 41020
02_rank_heatmap.png 60066
03_era_winrate.png 30617
04_playoff_boxplot.png 28272
05_transfer.png 24832
06_feature_importance.png 47645
07_playoff_prob.png 49264
08_apr_vs_final.png 28499
09_cluster_pca.png 31996

HTML: file:///C:/Users/Admin/Documents/GitHub/Ml_Baseball/notebooks/experiments/dy/kbo_2022_2026_master_dashboard.html
